In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

In [0]:
# jdbc_url = "jdbc:sqlserver://my-quant-server.database.windows.net:1433;database=qunat-db"
# user_name = "quant-user"
# password = "admin@123"

user_name = dbutils.secrets.get(scope="quant-secret-store", key="user-id")
password = dbutils.secrets.get(scope="quant-secret-store", key="db-password")

In [0]:
print(user_name)
print(password)

In [0]:
connection_properties = {
    "user": user_name,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

base_target_location = "/Volumes/quant_databricks/batch0506/quantcloudrawdatasets"

In [0]:
query = """
(select table_schema, table_name
from information_schema.tables
where table_type = 'BASE TABLE') as tmp
"""

In [0]:
tables_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", query)
    .options(**connection_properties)
    .load()
)

display(tables_df)

In [0]:
l = tables_df.collect()


In [0]:
# l[0][1]

In [0]:
def read_sql_table(table_schema: str, table_name: str) -> DataFrame:
    query = f"""
    (select * from {table_schema}.{table_name}) as tmp
    """
    return (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", query)
            .options(**connection_properties)
            .load()
    )

In [0]:
for table_info in tables_df.collect():
    table_schema = table_info[0]
    table_name = table_info[1]
    target_table = table_name + "_" + datetime.date.today().strftime("%Y%m%d")
    # print(table_schema, table_name)
    df = read_sql_table(table_schema, table_name)
    df.write.format("csv").mode("overwrite").option("header", "true").save(f"{base_target_location}/{target_table}")
    
